# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tab 1, Tab 2, Tab 3 Dual Fusion)
Triển khai trên Kaggle GPU (Tesla T4) + Cloudflare Tunnel (Băng thông KHÔNG GIỚI HẠN).
Tự động kết nối và đồng bộ 100% với Web App trên Render.com: https://frontendflood.onrender.com


In [ ]:
# 1. Cài đặt các thư viện cần thiết và tải Cloudflare Tunnel
!pip install -q "huggingface_hub<0.26.0" "gradio==4.44.0" "starlette<0.40.0" "fastapi<0.116.0" "pydantic<2.10.0" ultralytics opencv-python-headless pillow pandas lapx nest-asyncio
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared


In [ ]:
# 2. Đồng bộ mã nguồn Backend mới nhất từ GitHub SYNEXAN/FrontendFlood
import os, sys
!rm -rf /kaggle/working/FrontendFlood
!git clone https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())
!ls -lh


In [ ]:
# 3. Tự động tải Trọng số mô hình Deep Learning (Flood Seg & Vehicle Detection)
import os, urllib.request, time

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

for path, url, min_size in models_info:
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        continue
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path}.")
                break
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(2)

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/


In [ ]:
# 4. Khởi động Cloudflare Tunnel & Tự động kết nối trực tiếp về Render Frontend
import subprocess, time, os, re, urllib.request, json

RENDER_FRONTEND_URL = "https://frontendflood.onrender.com"
WEBHOOK_URL = "https://webhook.site/36a142f2-7175-4606-98f9-bcbad6d76837"

!pkill -f cloudflared
!pkill -f "python app.py"

if not os.path.exists("/tmp/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared

# Khởi động Cloudflare Tunnel trỏ vào Gradio port 7860
cf_cmd = ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860", "--logfile", "/tmp/cloudflared.log"]
cf_proc = subprocess.Popen(cf_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Chờ và đọc URL công khai từ file log Cloudflare
public_cf_url = None
for i in range(30):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            log_content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
        if match:
            public_cf_url = match.group(0)
            break

print("=" * 70)
print("🚀 SYNEXAN AI FLOOD & TRAFFIC BACKEND ĐÃ SẴN SÀNG TRÊN GPU TESLA T4!")
if public_cf_url:
    print(f"🔥 CLOUDFLARE PUBLIC URL: {public_cf_url}")
    # 1. Tự động gửi URL sang Render Frontend
    for attempt in range(1, 4):
        try:
            req = urllib.request.Request(
                f"{RENDER_FRONTEND_URL}/api/backend-url",
                data=json.dumps({"url": public_cf_url}).encode("utf-8"),
                headers={"Content-Type": "application/json"}
            )
            resp = urllib.request.urlopen(req, timeout=10)
            print(f"✅ ĐÃ TỰ ĐỘNG ĐỒNG BỘ URL VỀ RENDER: {RENDER_FRONTEND_URL}")
            print("🎉 Giao diện người dùng sẽ tự động hiển thị mà KHÔNG CẦN F5 hay copy/paste.")
            break
        except Exception as e:
            print(f"⚠️ Thử gửi Render lần {attempt}/3: {e}")
            time.sleep(2)
    # 2. Gửi tín hiệu về Webhook Site
    try:
        w_req = urllib.request.Request(WEBHOOK_URL, data=public_cf_url.encode("utf-8"), headers={"Content-Type": "text/plain"})
        urllib.request.urlopen(w_req, timeout=8)
        print("✅ Đã phát tín hiệu webhook thành công!")
    except Exception:
        pass
else:
    print("⚠️ Đang thiết lập Cloudflare... Nếu chưa hiện URL, hãy thử chạy lại cell.")
print("=" * 70)

# Chạy Backend Gradio App
!python app.py
